# CP2 · Examen blanc de la partie II — vérification de ta copie

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_2/02_examen_notebook.ipynb)

L'examen se fait **sur papier** (`01_examen_sujet.md`, 117 minutes, fiches fermées). Ce notebook ne sert qu'**après** : il vérifie les réponses chiffrées de ta copie. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`, et « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli (⏸️).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_2/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP2`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Après l'examen : vérification automatique

**Ne commence qu'une fois ta copie terminée**, examen rendu. Ce notebook vérifie les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Il ne note pas : les démarches, les justifications, le quiz CP2.Q1, la démonstration CP2.9 et les corrections de CP2.10 b) se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Reporte chaque réponse de ta feuille : **la valeur** que tu as écrite, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). Laisse `...` pour une question que tu n'as pas faite.
3. Les formes d'écriture : en Python, le séparateur décimal est un **point** (`2.75`) : une virgule arrête toute la cellule (`0,027`), et aucune réponse de la question n'est vérifiée, ou donne un couple de nombres (`0,75`) ; une liste s'écrit entre crochets (`[-1.5, 4]`), une matrice comme une liste de lignes (`[[1, -2], [0.5, 3]]`) ; un texte entre guillemets (`"XYZ"`) ; vrai ou faux par `True` ou `False`.
4. Le bilan, à la fin, dit aussi quelles réponses n'ont pas été vérifiées.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells below

RESULTS = {}        # sub-ID -> result of wbm.check
_REMINDED = []      # the full reminder is printed once, then a short one


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans cette cellule, puis exécute-la.")
        _REMINDED.append(True)
        return False
    if "wbm" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True

**CP2.1 — Normes, cosinus, projection et distance à un hyperplan**

In [ ]:
# CP2.1: the values written on your answer sheet
answer_CP2_1a = ...  # a) [norm 1, norm 2, norm inf]
answer_CP2_1b = ...  # b) a number
answer_CP2_1c = ...  # c) a number (3 decimals)
answer_CP2_1d = ...  # d) a list (2 decimals)
answer_CP2_1e = ...  # e) a number (3 decimals)
answer_CP2_1f = ...  # f) a number
answer_CP2_1g = ...  # g) a list (2 decimals)

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP2_1a, answer_CP2_1b, answer_CP2_1c, answer_CP2_1d, answer_CP2_1e, answer_CP2_1f, answer_CP2_1g]):
        RESULTS[f"CP2.1{letter}"] = wbm.check(f"CP2.1{letter}", answer)

**CP2.2 — Un test binomial et un intervalle de Wilson**

In [ ]:
# CP2.2: the values written on your answer sheet
answer_CP2_2a = ...  # a) a number (4 decimals)
answer_CP2_2b = ...  # b) a number (4 decimals)
answer_CP2_2c = ...  # c) a number (4 decimals)
answer_CP2_2d = ...  # d) True or False
answer_CP2_2e = ...  # e) an integer
answer_CP2_2f = ...  # f) [low, high] (4 decimals)
answer_CP2_2g = ...  # g) a number (4 decimals)

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP2_2a, answer_CP2_2b, answer_CP2_2c, answer_CP2_2d, answer_CP2_2e, answer_CP2_2f, answer_CP2_2g]):
        RESULTS[f"CP2.2{letter}"] = wbm.check(f"CP2.2{letter}", answer)

**CP2.3 — Comparaisons multiples, AIC et BIC**

In [ ]:
# CP2.3: the values written on your answer sheet
answer_CP2_3a = ...  # a) an integer
answer_CP2_3b = ...  # b) an integer
answer_CP2_3c = ...  # c) a number (3 decimals)
answer_CP2_3d = ...  # d) a list (1 decimal)
answer_CP2_3e = ...  # e) a list (1 decimal)
answer_CP2_3f = ...  # f) [AIC choice, BIC choice]

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP2_3a, answer_CP2_3b, answer_CP2_3c, answer_CP2_3d, answer_CP2_3e, answer_CP2_3f]):
        RESULTS[f"CP2.3{letter}"] = wbm.check(f"CP2.3{letter}", answer)

**CP2.4 — Pivot de Gauss, rang et inverse**

In [ ]:
# CP2.4: the values written on your answer sheet
answer_CP2_4a = ...  # a) a list of 4 numbers
answer_CP2_4b = ...  # b) a list of 4 numbers
answer_CP2_4c = ...  # c) [x, y, z]
answer_CP2_4d = ...  # d) [[a, b], [c, d]]
answer_CP2_4e = ...  # e) a list of 2 numbers
answer_CP2_4f = ...  # f) an integer
answer_CP2_4g = ...  # g) a number

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP2_4a, answer_CP2_4b, answer_CP2_4c, answer_CP2_4d, answer_CP2_4e, answer_CP2_4f, answer_CP2_4g]):
        RESULTS[f"CP2.4{letter}"] = wbm.check(f"CP2.4{letter}", answer)

**CP2.5 — Les équations normales d'un petit modèle, avec et sans ridge**

In [ ]:
# CP2.5: the values written on your answer sheet
answer_CP2_5a = ...  # a) [[a, b], [c, d]]
answer_CP2_5b = ...  # b) a list of 2 numbers
answer_CP2_5c = ...  # c) a list (4 decimals)
answer_CP2_5d = ...  # d) a list (4 decimals)
answer_CP2_5e = ...  # e) a number (4 decimals)
answer_CP2_5f = ...  # f) a list (4 decimals)
answer_CP2_5g = ...  # g) a number (2 decimals)

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP2_5a, answer_CP2_5b, answer_CP2_5c, answer_CP2_5d, answer_CP2_5e, answer_CP2_5f, answer_CP2_5g]):
        RESULTS[f"CP2.5{letter}"] = wbm.check(f"CP2.5{letter}", answer)

**CP2.6 — Biais, variance et erreur quadratique d'un estimateur**

In [ ]:
# CP2.6: the values written on your answer sheet
answer_CP2_6a = ...  # a) a number (2 decimals)
answer_CP2_6b = ...  # b) a number (4 decimals)
answer_CP2_6c = ...  # c) a number (4 decimals)
answer_CP2_6d = ...  # d) a number (4 decimals)
answer_CP2_6e = ...  # e) a number (4 decimals)
answer_CP2_6f = ...  # f) 1 or 2

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP2_6a, answer_CP2_6b, answer_CP2_6c, answer_CP2_6d, answer_CP2_6e, answer_CP2_6f]):
        RESULTS[f"CP2.6{letter}"] = wbm.check(f"CP2.6{letter}", answer)

**CP2.7 — Convexité, enveloppe convexe et marge**

In [ ]:
# CP2.7: the values written on your answer sheet
answer_CP2_7a = ...  # a) [[x, y], ...]
answer_CP2_7b = ...  # b) a number
answer_CP2_7c = ...  # c) True or False
answer_CP2_7d = ...  # d) a number (3 decimals)
answer_CP2_7e = ...  # e) a number (3 decimals)
answer_CP2_7f = ...  # f) [w1, w2, b]

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP2_7a, answer_CP2_7b, answer_CP2_7c, answer_CP2_7d, answer_CP2_7e, answer_CP2_7f]):
        RESULTS[f"CP2.7{letter}"] = wbm.check(f"CP2.7{letter}", answer)

**CP2.8 — Markov, Tchebychev et Hoeffding**

In [ ]:
# CP2.8: the values written on your answer sheet
answer_CP2_8a = ...  # a) a number
answer_CP2_8b = ...  # b) a number (4 decimals)
answer_CP2_8c = ...  # c) an integer
answer_CP2_8d = ...  # d) a number (4 decimals)
answer_CP2_8e = ...  # e) a number (4 decimals)
answer_CP2_8f = ...  # f) an integer

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP2_8a, answer_CP2_8b, answer_CP2_8c, answer_CP2_8d, answer_CP2_8e, answer_CP2_8f]):
        RESULTS[f"CP2.8{letter}"] = wbm.check(f"CP2.8{letter}", answer)

**CP2.10 — Corriger une copie : cinq erreurs (les lettres seulement)**

In [ ]:
# CP2.10: the values written on your answer sheet
answer_CP2_10a = ...  # a) the letters, as text (like "ABCDE")

if exam_over():
    for letter, answer in zip("a", [answer_CP2_10a]):
        RESULTS[f"CP2.10{letter}"] = wbm.check(f"CP2.10{letter}", answer)

**CP2.11 — Problème de synthèse : ajuster un modèle, mesurer son erreur, décider**

In [ ]:
# CP2.11: the values written on your answer sheet
answer_CP2_11a = ...  # a) [sum of x², sum of x y]
answer_CP2_11b = ...  # b) [a, b] (4 decimals)
answer_CP2_11c = ...  # c) a number (2 decimals)
answer_CP2_11d = ...  # d) a number (3 decimals)
answer_CP2_11e = ...  # e) a number (4 decimals)
answer_CP2_11f = ...  # f) a number (1 decimal)
answer_CP2_11g = ...  # g) 0, 1 or 2
answer_CP2_11h = ...  # h) a number (4 decimals)
answer_CP2_11i = ...  # i) True or False

if exam_over():
    for letter, answer in zip("abcdefghi", [answer_CP2_11a, answer_CP2_11b, answer_CP2_11c, answer_CP2_11d, answer_CP2_11e, answer_CP2_11f, answer_CP2_11g, answer_CP2_11h, answer_CP2_11i]):
        RESULTS[f"CP2.11{letter}"] = wbm.check(f"CP2.11{letter}", answer)

## Bilan

In [ ]:
# Summary of the automatic checks
import re

EXPECTED = ['CP2.1a', 'CP2.1b', 'CP2.1c', 'CP2.1d', 'CP2.1e', 'CP2.1f', 'CP2.1g', 'CP2.2a', 'CP2.2b', 'CP2.2c', 'CP2.2d', 'CP2.2e', 'CP2.2f', 'CP2.2g', 'CP2.3a', 'CP2.3b', 'CP2.3c', 'CP2.3d', 'CP2.3e', 'CP2.3f', 'CP2.4a', 'CP2.4b', 'CP2.4c', 'CP2.4d', 'CP2.4e', 'CP2.4f', 'CP2.4g', 'CP2.5a', 'CP2.5b', 'CP2.5c', 'CP2.5d', 'CP2.5e', 'CP2.5f', 'CP2.5g', 'CP2.6a', 'CP2.6b', 'CP2.6c', 'CP2.6d', 'CP2.6e', 'CP2.6f', 'CP2.7a', 'CP2.7b', 'CP2.7c', 'CP2.7d', 'CP2.7e', 'CP2.7f', 'CP2.8a', 'CP2.8b', 'CP2.8c', 'CP2.8d', 'CP2.8e', 'CP2.8f', 'CP2.10a', 'CP2.11a', 'CP2.11b', 'CP2.11c', 'CP2.11d', 'CP2.11e', 'CP2.11f', 'CP2.11g', 'CP2.11h', 'CP2.11i']


def natural(sub_id):
    """CP2.2a before CP2.10a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP2\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    missing = [k for k in EXPECTED if k not in RESULTS]
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    if missing:
        print("⚠️ Pas vérifiées (cellule pas exécutée, ou arrêtée par une erreur, par exemple une virgule décimale) :",
              ", ".join(missing))
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; le quiz CP2.Q1 ; la démonstration CP2.9 ; les corrections de CP2.10 b) (seules les lettres de a) sont vérifiées). Tout cela se corrige avec `03_examen_corrige.md`. Remplis ensuite le tableau de ta note, à la fin de ta copie, puis fais la remédiation du corrigé.